In [5]:
pip install xarray netCDF4 pandas sentence-transformers faiss-cpu

  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
    --------------------------------------- 0.0/1.3 MB 1.3 MB/s eta 0:00:02
   ----- ---------------------------------- 0.2/1.3 MB 2.9 MB/s eta 0:00:01
   -------------- ------------------------- 0.5/1.3 MB 4.2 MB/s eta 0:00:01
   ------------------------------ --------- 1.0/1.3 MB 6.4 MB/s eta 0:00:01
   ---------------------------------------  1.3/1.3 MB 7.1 MB/s eta 0:00:01
   ---------------------------------------- 1.3/1.3 MB 6.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/7.0 MB ? eta -:--:--
   --- ------------------------------------ 0.6/7.0 MB 18.2 MB/s eta 0:00:01
   ----- ---------------------------------- 0.9/7.0 MB 14.3 MB/s eta 0:00:01
   ----- ---------------------------------- 1.0/7.0 MB 13.3 MB/s eta 0:00:01
   ----- ---------------------------------


[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import pandas as pd
import xarray as xr
import sentence_transformers as st
import faiss

c:\Users\Admin\Documents\PROJECTS\SIH\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
from pathlib import Path
import json
import numpy as np

FN = Path(r"c:\Users\Admin\Documents\PROJECTS\SIH\data\1900121_meta.nc")

def safe_str(x):
    try:
        if isinstance(x, bytes):
            return x.decode("utf-8", "ignore")
        return str(x)
    except Exception:
        return repr(x)

def decode_char_array(arr):
    # handle char arrays stored as bytes or as last-dimension characters
    try:
        arr = np.asarray(arr)
        if arr.dtype.kind in ("S", "a", "O", "U"):
            # If last axis is characters, join them
            if arr.ndim > 1 and arr.shape[-1] <= 1024:
                try:
                    joined = np.apply_along_axis(
                        lambda row: b"".join([c if isinstance(c, (bytes, bytearray)) else bytes(str(c), "utf-8") for c in row]).decode("utf-8", "ignore"),
                        axis=-1,
                        arr=arr
                    )
                    return joined
                except Exception:
                    pass
            # fallback: decode elements
            vectorized = np.vectorize(lambda v: safe_str(v))
            return vectorized(arr)
        else:
            return arr
    except Exception:
        return arr

def inventory_nc(path):
    inv = {"path": str(path), "dimensions": {}, "global_attributes": {}, "variables": {}}
    # try xarray first
    try:
        import xarray as xr
        ds = xr.open_dataset(path, decode_cf=True, decode_times=False)
        inv["dimensions"] = dict(ds.dims)
        inv["global_attributes"] = {k: safe_str(v) for k, v in ds.attrs.items()}
        for name in ds.variables:
            var = ds.variables[name]
            try:
                vals = var.values
                vals = decode_char_array(vals)
                # sample
                if hasattr(vals, "size") and vals.size > 20:
                    if hasattr(vals, "flatten"):
                        sample = vals.flatten()[:20].tolist()
                    else:
                        sample = str(vals)[:200]
                else:
                    sample = vals.tolist() if hasattr(vals, "tolist") else safe_str(vals)
            except Exception as e:
                sample = "<unreadable: %s>" % e
            inv["variables"][name] = {
                "dtype": str(var.dtype),
                "shape": tuple(var.shape),
                "attributes": {k: safe_str(v) for k, v in var.attrs.items()},
                "sample": sample
            }
        ds.close()
        return inv
    except Exception:
        # fallback to netCDF4
        from netCDF4 import Dataset
        ds = Dataset(path)
        inv["dimensions"] = {k: len(ds.dimensions[k]) for k in ds.dimensions}
        inv["global_attributes"] = {a: safe_str(getattr(ds, a)) for a in ds.ncattrs()}
        for name in ds.variables:
            v = ds.variables[name]
            try:
                vals = v[:]
                vals = decode_char_array(vals)
                if hasattr(vals, "size") and vals.size > 20:
                    sample = np.asarray(vals).flatten()[:20].tolist()
                else:
                    sample = vals.tolist() if hasattr(vals, "tolist") else safe_str(vals)
            except Exception as e:
                sample = "<unreadable: %s>" % e
            inv["variables"][name] = {
                "dtype": str(v.datatype),
                "shape": tuple(v.shape),
                "attributes": {a: safe_str(getattr(v, a)) for a in v.ncattrs()},
                "sample": sample
            }
        ds.close()
        return inv

if __name__ == "__main__":
    if not FN.exists():
        raise SystemExit("File not found: " + str(FN))
    inv = inventory_nc(FN)
    out = FN.with_name(FN.stem + "_inventory.json")
    out.write_text(json.dumps(inv, indent=2, ensure_ascii=False))
    print("Wrote inventory to:", out)
    # print short summary
    print("Dimensions:", inv["dimensions"])
    print("Variables:", list(inv["variables"].keys()))
    print("Global attrs keys:", list(inv["global_attributes"].keys()))

Wrote inventory to: c:\Users\Admin\Documents\PROJECTS\SIH\data\1900121_meta_inventory.json
Dimensions: {'N_TRANS_SYSTEM': 1, 'N_POSITIONING_SYSTEM': 1, 'N_LAUNCH_CONFIG_PARAM': 14, 'N_CONFIG_PARAM': 14, 'N_MISSIONS': 1, 'N_SENSOR': 3, 'N_PARAM': 3}
Variables: ['DATA_TYPE', 'FORMAT_VERSION', 'HANDBOOK_VERSION', 'DATE_CREATION', 'DATE_UPDATE', 'PLATFORM_NUMBER', 'PTT', 'TRANS_SYSTEM', 'TRANS_SYSTEM_ID', 'TRANS_FREQUENCY', 'POSITIONING_SYSTEM', 'PLATFORM_FAMILY', 'PLATFORM_TYPE', 'PLATFORM_MAKER', 'FIRMWARE_VERSION', 'MANUAL_VERSION', 'FLOAT_SERIAL_NO', 'STANDARD_FORMAT_ID', 'DAC_FORMAT_ID', 'WMO_INST_TYPE', 'PROJECT_NAME', 'DATA_CENTRE', 'PI_NAME', 'ANOMALY', 'BATTERY_TYPE', 'BATTERY_PACKS', 'CONTROLLER_BOARD_TYPE_PRIMARY', 'CONTROLLER_BOARD_TYPE_SECONDARY', 'CONTROLLER_BOARD_SERIAL_NO_PRIMARY', 'CONTROLLER_BOARD_SERIAL_NO_SECONDARY', 'SPECIAL_FEATURES', 'FLOAT_OWNER', 'OPERATING_INSTITUTION', 'CUSTOMISATION', 'LAUNCH_DATE', 'LAUNCH_LATITUDE', 'LAUNCH_LONGITUDE', 'LAUNCH_QC', 'START_DATE

C:\Users\Admin\AppData\Local\Temp\ipykernel_14176\1253431491.py:45: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  inv["dimensions"] = dict(ds.dims)


In [11]:
# ...existing code...
# Add a notebook cell after the inventory code to build and show tables

from IPython.display import display, HTML
import pandas as pd
import json

# ensure inventory exists
try:
    inv
except NameError:
    inv = inventory_nc(FN)

# build variables table
rows = []
for name, info in inv["variables"].items():
    attrs = info.get("attributes", {})
    # stringify attributes (truncate long strings)
    attrs_str = json.dumps(attrs, ensure_ascii=False)
    if len(attrs_str) > 300:
        attrs_str = attrs_str[:300] + "..."
    sample = info.get("sample")
    sample_str = json.dumps(sample, ensure_ascii=False) if not isinstance(sample, str) else sample
    if len(sample_str) > 200:
        sample_str = sample_str[:200] + "..."
    rows.append({
        "variable": name,
        "dtype": info.get("dtype"),
        "shape": str(info.get("shape")),
        "attributes": attrs_str,
        "sample": sample_str
    })

df_vars = pd.DataFrame(rows).sort_values("variable").reset_index(drop=True)

# dimensions table
df_dims = pd.DataFrame(list(inv.get("dimensions", {}).items()), columns=["dimension", "length"])

# global attributes table
df_globals = pd.DataFrame(list(inv.get("global_attributes", {}).items()), columns=["attribute", "value"])

# display nicely in notebook
display(HTML("<h3>Variables</h3>"))
display(df_vars.style.set_properties(**{"white-space": "pre-wrap"}))
display(HTML("<h3>Dimensions</h3>"))
display(df_dims)
display(HTML("<h3>Global attributes</h3>"))
display(df_globals)

# save CSV/HTML outputs alongside .nc file
out_dir = FN.parent
df_vars.to_csv(out_dir / f"{FN.stem}_variables.csv", index=False)
df_dims.to_csv(out_dir / f"{FN.stem}_dimensions.csv", index=False)
df_globals.to_csv(out_dir / f"{FN.stem}_global_attributes.csv", index=False)
(out_dir / f"{FN.stem}_variables.html").write_text(df_vars.to_html(index=False, escape=False))

print("Saved CSV/HTML files to", out_dir)

,variable,dtype,shape,attributes,sample
0,ANOMALY,object,(),"{""long_name"": ""Describe any anomalies or problems the float may have had""}",...
1,BATTERY_PACKS,object,(),"{""long_name"": ""Configuration of battery packs in the float""}",3D Alk + 1C Alk
2,BATTERY_TYPE,object,(),"{""long_name"": ""Type of battery packs in the float""}",Alkaline
3,CONFIG_MISSION_COMMENT,object,"(1,)","{""long_name"": ""Comment on configuration""}","["" ..."
4,CONFIG_MISSION_NUMBER,float64,"(1,)","{""long_name"": ""Unique number denoting the missions performed by the float"", ""conventions"": ""1...N, 1 : first complete mission""}",[1.0]
5,CONFIG_PARAMETER_NAME,object,"(14,)","{""long_name"": ""Name of configuration parameter""}","[""CONFIG_TransmissionRepetitionPeriod_seconds "", ""CONFIG_ParkTime_hours ..."
6,CONFIG_PARAMETER_VALUE,float64,"(1, 14)","{""long_name"": ""Value of configuration parameter""}","[[90.0, 221.0, 9.5, 9.5, 2000.0, 19.0, 2000.0, 6.0, 221.0, 1.0, 240.0, 30.0, 25.0, 47.0]]"
7,CONTROLLER_BOARD_SERIAL_NO_PRIMARY,object,(),"{""long_name"": ""Serial number of the primary controller board""}",8c-1082
8,CONTROLLER_BOARD_SERIAL_NO_SECONDARY,object,(),"{""long_name"": ""Serial number of the secondary controller board""}",
9,CONTROLLER_BOARD_TYPE_PRIMARY,object,(),"{""long_name"": ""Type of primary controller board""}",APF8


,dimension,length
0,N_TRANS_SYSTEM,1
1,N_POSITIONING_SYSTEM,1
2,N_LAUNCH_CONFIG_PARAM,14
3,N_CONFIG_PARAM,14
4,N_MISSIONS,1
5,N_SENSOR,3
6,N_PARAM,3


,attribute,value
0,title,Argo float metadata file
1,institution,INCOIS
2,source,Argo float
3,history,2015-10-12T05:38:41Z creation;2015-10-13T05:54...
4,references,http://www.argodatamgt.org/Documentation
5,user_manual_version,3.1
6,Conventions,Argo-3.1 CF-1.6


Saved CSV/HTML files to c:\Users\Admin\Documents\PROJECTS\SIH\data
